# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, find_all_hrs_ft_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
    FT_SNAP_HZ, ft_snap_hz, ft_stim_adc_hz, compute_ft_trial_hz,
    load_all_recordings, make_ft_viewer,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [ ]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    

    #("HRPILOT-17 CM1",  "HRPilot-17_Control/BASELINE1_HRPILOT-17_BOOTH1_250US_10KHZ_8-20-26",        10000.0),
    #("HRPILOT-17 CM2",  "HRPilot-17_Control/BASELINE2_HRPILOT-17_BOOTH1_250US_10KHZ_8-21-26",        10000.0),
    #("HRPILOT-17 CM3",  "HRPilot-17_Control/BASELINE3_HRPILOT-17_BOOTH1_250US_10KHZ_8-24-26",        10000.0),
    #("HRPILOT-17 CM4",  "HRPilot-17_Control/BASELINE4_HRPILOT-17_BOOTH1_250US_10KHZ_8-25-26",        10000.0),
    #("HRPILOT-17 CM5",  "HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26",        10000.0),
    #("HRPILOT-17 CM6",  "HRPilot-17_Control/BASELINE6_HRPILOT-17_BOOTH1_250US_10KHZ_8-27-26",        10000.0),
    #("HRPILOT-17 CM7",  "HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26",        10000.0),
    
    
    # Frequency Tests
    #("HRPILOT-17 FT1",  "HRPilot-17_Control/CCC1_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26",        10000.0),
    #("HRPILOT-17 FT2",  "HRPilot-17_Control/CCC2_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26",        10000.0),
    #("HRPILOT-17 Offline FT3",  "HRPilot-17_Control/OFFLINE_FREQTEST1_HRPILOT-17_BOOTH1_10KHZ_250US_8-14-26",        10000.0),
    #("HRPILOT-17 Online FT4",  "HRPilot-17_Control/Old/ONLINE_FREQTEST1_HRPILOT-17_BOOTH1_10KHZ_250US_8-14-26",        10000.0),
        
    
    # Upconditioning days
    #("HRPILOT-17 UPC1",  "HRPilot-17_Up_Cond_Pellet/UPCP1_HRPILOT-17_250US_10KHZ_8-31-26",              10000.0),
    #("HRPILOT-17 UPC2",  "HRPilot-17_Up_Cond_Pellet/UPCP2_HRPILOT-17_250US_10KHZ_9-1-26",               10000.0),
    #("HRPILOT-17 UPC3",  "HRPilot-17_Up_Cond_Pellet/UPCP3_HRPILOT-17_250US_10KHZ_9-2-26",               10000.0),
    #("HRPILOT-17 UPC4",  "HRPilot-17_Up_Cond_Pellet/UPCP4_HRPILOT-17_BOOTH1_250US_10KHZ_9-3-26",        10000.0),
    #("HRPILOT-17 UPC5",  "HRPilot-17_Up_Cond_Pellet/UPCP5_HRPILOT-17_BOOTH1_500US_10KHZ_9-4-26",        10000.0),
    #("HRPILOT-17 UPC6",  "HRPilot-17_Up_Cond_Pellet/UPCP6_HRPILOT-17_BOOTH1_500US_10KHZ_9-8-26",        10000.0),
    
    
    # Add more recordings below — uncomment or append new tuples.
    # HRPilot-23 Recordings
    #("HRPILOT-23 250US",  "Calibration/CALIB1_HRPILOT-23_BOOTH2_250US_10KHZ_8-31-26",        10000.0),
    #("HRPILOT-23 100US",  "Calibration/CALIB2_HRPILOT-23_BOOTH3_100US_10KHZ_9-2-26",        10000.0),
    #("FT1 HRPILOT-23 250US",  "Calibration/FT1_HRPILOT-23_100US_10KHZ_9-8-26",        10000.0),
    ("Calib3 HRPILOT-23 100US",  "Calibration/CALIB3_HRPILOT-23_BOOTH1_100US_10KHZ_9-10-26",        10000.0),
    
    
    # HRPilot-25 Recordings
    #("HRPILOT-25 250US",  "Calibration/CALIB1_HRPILOT-25_BOOTH2_250US_10KHZ_8-25-26",        10000.0),
    #("HRPILOT-25 100US",  "Calibration/CALIB2_HRPILOT-25_BOOTH2_100US_10KHZ_9-2-26",        10000.0),
    ("Calib3 HRPILOT-25 250US",  "Calibration/CALIB3_HRPILOT-25_BOOTH1_250US_10KHZ_9-10-26",        10000.0),
    
    # HRPilot-26 Recordings
    #("HRPILOT-26 250US",  "Calibration/CALIB1_HRPILOT-26_BOOTH2_250US_10KHZ_8-27-26",        10000.0),
    #("HRPILOT-26 100US",  "Calibration/CALIB2_HRPILOT-26_BOOTH1_100US_10KHZ_9-2-26",        10000.0),
    
    # HRPilot-34 Recordings
    #("HRPILOT-34 250US",  "Calibration/CALIB1_HRPILOT-34_BOOTH2_250US_10KHZ_9-2-26",        10000.0),
    ("Calib2 HRPILOT-34 250US",  "Calibration/CALIB2_HRPILOT-34_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    
    # HRPilot-36 Recordings
    #("HRPILOT-36 250US",  "Calibration/CALIB1_HRPILOT-36_BOOTH3_250US_10KHZ_9-1-26",        10000.0),
    ("Calib2 HRPILOT-36 250US",  "Calibration/CALIB2_HRPILOT-36_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    
    
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

0 recording(s) configured.


In [3]:
# ── Load all recordings (auto-detects V2 / V3) ─────────────────────────────────
# Customize FT_SNAP_HZ to match your experiment's pulse-train frequencies (Hz).
# Values within ±25% of a target snap to that target.
FT_SNAP_HZ_CUSTOM = FT_SNAP_HZ  # use [5.0, 10.0, 15.0, 20.0, 33.0] or override here

_all_recordings = load_all_recordings(RECORDING_DIRS, ft_snap_hz_list=FT_SNAP_HZ_CUSTOM)
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')

StopIteration: 

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [ ]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'HRPILOT-17 UPC6'
  MH Recruitment Curve (.hrs1): 1242 trials
  Up Condition Pellet (.hrs4): 1 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [ ]:
# ── Recording & Stage Selector ─────────────────────────────────────────────────
# Change the active recording/stage here — all viewer sections update automatically.
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

# ── Shared viewer callback registry ────────────────────────────────────────────
if not isinstance(globals().get('_refresh_viewers'), dict):
    _refresh_viewers = {}

def _trigger_refresh():
    for _fn in list(_refresh_viewers.values()):
        try:
            _fn()
        except Exception as _err:
            import traceback
            print(f'[viewer refresh error] {_err}')
            traceback.print_exc()

# ── Active-recording state (module-level vars used by downstream cells) ────────
def _activate_recording(label):
    global _stage_map, recording_sample_rate, hrs1_header, \
           ft_trials, ft_header, ft_files, ACTIVE_STAGE, \
           _plot_trials, _plot_header, _plot_emg_blocks
    _rec = _all_recordings[label]
    _stage_map            = _rec['stage_map']
    recording_sample_rate = _rec['sample_rate']
    hrs1_header           = _rec['hrs1_header']
    ft_trials             = _rec.get('ft_trials')
    ft_header             = _rec.get('ft_header')
    ft_files              = _rec.get('ft_files', {})
    if globals().get('ACTIVE_STAGE') not in _stage_map:
        ACTIVE_STAGE = next(iter(_stage_map)) if _stage_map else None
    if ACTIVE_STAGE and ACTIVE_STAGE in _stage_map:
        _sel = _stage_map[ACTIVE_STAGE]
        _plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]

_activate_recording(_active_rec_label)

# ── Global stage dropdown ──────────────────────────────────────────────────────
_sd_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
_stage_drop = Dropdown(options=_sd_opts, value=ACTIVE_STAGE,
                       description='Stage:', layout={'width': '480px'})

def _on_stage_change(change):
    global ACTIVE_STAGE, _plot_trials, _plot_header, _plot_emg_blocks
    ACTIVE_STAGE = _stage_drop.value
    _sel = _stage_map[ACTIVE_STAGE]
    _plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]
    _trigger_refresh()

_stage_drop.observe(_on_stage_change, names='value')

if len(_all_recordings) > 1:
    _rec_drop = Dropdown(options=list(_all_recordings.keys()),
                         value=_active_rec_label,
                         description='Recording:', layout={'width': '640px'})

    def _on_rec_change(change):
        global _active_rec_label
        _active_rec_label = _rec_drop.value
        _activate_recording(_active_rec_label)
        _stage_drop.unobserve(_on_stage_change, names='value')
        _stage_drop.options = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
        _stage_drop.value   = ACTIVE_STAGE
        _stage_drop.observe(_on_stage_change, names='value')
        _trigger_refresh()

    _rec_drop.observe(_on_rec_change, names='value')
    _disp(VBox([_rec_drop, _stage_drop]))
else:
    _disp(_stage_drop)

# ── Status ─────────────────────────────────────────────────────────────────────
print(f'Active recording : {_active_rec_label!r}  (App V{_all_recordings[_active_rec_label]["app_version"]})')
print(f'Available stages ({len(_stage_map)}):')
for _k, (_t, _h, _e, _lbl) in _stage_map.items():
    _mark = '  ◄ active' if _k == ACTIVE_STAGE else ''
    print(f'  {_k!r:<22} → {_lbl}  ({len(_t)} trials){_mark}')
print(f'Sample rate : {recording_sample_rate} Hz')

# ── Helper: local Recording + Stage selector for any viewer ───────────────────
def _make_viewer_selector(refresh_fn):
    """Create independent Recording + Stage dropdowns for a viewer section.

    Returns (ctrl_widget, get_state_fn, sync_fn).

    get_state_fn() -> (stage_key, trials, header, emg, label, sample_rate, rec_label, hrs1_hdr)
    sync_fn()      -> called by the global selector to mirror its current state here.
    """
    from ipywidgets import Dropdown, VBox

    def _stage_opts(rec_label):
        sm = _all_recordings[rec_label]['stage_map']
        return [(lbl, sk) for sk, (_t, _h, _e, lbl) in sm.items() if _t]

    _local_rec   = [_active_rec_label]
    _local_stage = [ACTIVE_STAGE]

    _sd = Dropdown(options=_stage_opts(_active_rec_label),
                   value=ACTIVE_STAGE,
                   description='Stage:', layout={'width': '480px'})

    def _on_stage(change):
        _local_stage[0] = _sd.value
        refresh_fn()

    _sd.observe(_on_stage, names='value')

    if len(_all_recordings) > 1:
        _rd = Dropdown(options=list(_all_recordings.keys()),
                       value=_active_rec_label,
                       description='Recording:', layout={'width': '640px'})

        def _on_rec(change):
            _local_rec[0] = _rd.value
            _sd.unobserve(_on_stage, names='value')
            opts = _stage_opts(_rd.value)
            _sd.options = opts
            sm = _all_recordings[_rd.value]['stage_map']
            _sd.value = _local_stage[0] if _local_stage[0] in sm else (opts[0][1] if opts else None)
            _local_stage[0] = _sd.value
            _sd.observe(_on_stage, names='value')
            refresh_fn()

        _rd.observe(_on_rec, names='value')
        ctrl = VBox([_rd, _sd])
    else:
        _rd = None
        ctrl = VBox([_sd])

    def _get_state():
        rec = _local_rec[0]
        sk  = _local_stage[0]
        sm  = _all_recordings[rec]['stage_map']
        if sk not in sm:
            sk = next(iter(sm))
        st, sh, se, slbl = sm[sk]
        sr = _all_recordings[rec]['sample_rate']
        h1 = _all_recordings[rec]['hrs1_header']
        return sk, st, sh, se, slbl, sr, rec, h1

    def _sync():
        """Push global selector state into this viewer's local dropdowns."""
        if _rd is not None:
            _rd.unobserve(_on_rec, names='value')
            _rd.value = _active_rec_label
            _local_rec[0] = _active_rec_label
            _rd.observe(_on_rec, names='value')
        _sd.unobserve(_on_stage, names='value')
        opts = _stage_opts(_active_rec_label)
        _sd.options = opts
        sm = _all_recordings[_active_rec_label]['stage_map']
        _sd.value = ACTIVE_STAGE if ACTIVE_STAGE in sm else (opts[0][1] if opts else None)
        _local_stage[0] = _sd.value
        _sd.observe(_on_stage, names='value')
        refresh_fn()

    return ctrl, _get_state, _sync

# ── Stimulation Intensity Histogram ───────────────────────────────────────────
_hist_out = Output()

def _hist_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _hist_get_state()
    with _hist_out:
        _hist_out.clear_output(wait=True)
        print(f'\nHistogram: {_slbl}  ({len(_st)} trials)  [{_rec}]')
        plot_amplitude_distribution(_st, _sh)

_hist_ctrl, _hist_get_state, _hist_sync = _make_viewer_selector(_hist_refresh)
_refresh_viewers['histogram'] = _hist_sync
_disp(VBox([_hist_ctrl, _hist_out]))
_hist_refresh()


Dropdown(description='Stage:', index=1, layout=Layout(width='480px'), options=(('MH Recruitment Curve (.hrs1)'…

Active recording : 'HRPILOT-17 UPC6'  (App V3)
Available stages (2):
  'mh_recruitment'       → MH Recruitment Curve (.hrs1)  (1242 trials)
  'up_cond_pellet'       → Up Condition Pellet (.hrs4)  (1 trials)  ◄ active
Sample rate : 10000.0 Hz


In [ ]:
# ── Trial Timeline ─────────────────────────────────────────────────────────────
from ipywidgets import Output, VBox
from IPython.display import display as _disp

_tl_out = Output()

def _tl_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _tl_get_state()
    with _tl_out:
        _tl_out.clear_output(wait=True)
        print(f'\n── Trial Timeline: {_slbl}  ({len(_st)} trials)  [{_rec}]')
        plot_actual_trial_timeline(_st, header=_sh)

_tl_ctrl, _tl_get_state, _tl_sync = _make_viewer_selector(_tl_refresh)
_refresh_viewers['trial_timeline'] = _tl_sync
_disp(VBox([_tl_ctrl, _tl_out]))
_tl_refresh()


# Section 3b: "Most Recent Background" + "Background EMG Level"

Recreates the H-Reflex App recruitment-curve trial-plot widgets from `MhRecruitmentCurveStage.get_trial_plot_options`:

- **Most recent background** bar chart of the pre-stim |EMG| bins.
- **EMG Level** scatter of background grand means across trials.

Bins are reconstructed from `hrs2_emg_blocks` over a fixed monitoring window (default 2500 ms ending at trigger time).

In [ ]:
# ── Background EMG Views ──────────────────────────────────────────────────────
from ipywidgets import Output, VBox
from IPython.display import display as _disp

_bg_out = Output()

def _bg_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _bg_get_state()
    with _bg_out:
        _bg_out.clear_output(wait=True)
        print(f'\n── Background EMG: {_slbl}  ({len(_st)} trials)  [{_rec}]')
        plot_background_emg_views(_st, _se, monitoring_window_ms=2500)

_bg_ctrl, _bg_get_state, _bg_sync = _make_viewer_selector(_bg_refresh)
_refresh_viewers['background_emg'] = _bg_sync
_disp(VBox([_bg_ctrl, _bg_out]))
_bg_refresh()


In [ ]:
#  Configuration 
PRE_PLOT_MS  = 2   # ms before stim onset to display
POST_PLOT_MS = 15  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
M_WAVE_START_MS = 2.2
M_WAVE_END_MS   = 4.2
H_WAVE_START_MS = 5
H_WAVE_END_MS   = 9.6

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [ ]:
'''# ---- Failed Trial Detector & Corrected Trial Windowing ----
# Classifies all trials for ADC-sync failures, realigns each failed trial to the
# true stim onset found via the first ADC pulse in the continuous context window,
# and plots original (gray) vs corrected (black) waveforms.
# Returns trial_report, failed, passed, realigned for use in later cells.
trial_report, failed, passed, realigned = detect_and_correct_failed_trials(
    _plot_trials, _plot_header, _plot_emg_blocks,
    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)'''

'# ---- Failed Trial Detector & Corrected Trial Windowing ----\n# Classifies all trials for ADC-sync failures, realigns each failed trial to the\n# true stim onset found via the first ADC pulse in the continuous context window,\n# and plots original (gray) vs corrected (black) waveforms.\n# Returns trial_report, failed, passed, realigned for use in later cells.\ntrial_report, failed, passed, realigned = detect_and_correct_failed_trials(\n    _plot_trials, _plot_header, _plot_emg_blocks,\n    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,\n    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,\n    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,\n    sample_rate=recording_sample_rate or hrs1_header.sample_rate,\n)'

In [ ]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = False   # True → collapse every amplitude into one group
MERGED_GROUPS = []

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = False  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [ ]:
# ── Pre-compute H-Reflex Comparison Data ──────────────────────────────────────────────
# Computed once here for instant rendering in the comparison plot below.
# Re-run this cell if you change PRE_AVG_MS, POST_AVG_MS, or H/M-wave window constants.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'H-reflex comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

H-reflex comparison data pre-computed for 1 recording(s), 2 stage(s): ['mh_recruitment', 'up_cond_pellet']


In [ ]:
# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────────
from ipywidgets import Output, VBox
from IPython.display import display as _disp

_ana_out = Output()

def _ana_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _ana_get_state()
    if _sk == 'frequency_test':
        with _ana_out:
            _ana_out.clear_output(wait=True)
            print('HRS2 Analysis is not available for Frequency Test (.hrsft) stages.\n'
                  'Use the Frequency Test viewer section below.')
        return
    _tp = _apply_merge(_st)
    with _ana_out:
        _ana_out.clear_output(wait=True)
        print(f'\n── Analysis: {_slbl}  ({len(_st)} trials)  [{_rec}]')
        plot_hrs2_analysis(
            _tp, _sh,
            pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
            n_per_page=N_PER_PAGE,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=_sr,
            emg_blocks=_se,
        )

_ana_ctrl, _ana_get_state, _ana_sync = _make_viewer_selector(_ana_refresh)
_refresh_viewers['hrs2_analysis'] = _ana_sync
_disp(VBox([_ana_ctrl, _ana_out]))
_ana_refresh()


# Section 6b: H-Reflex Size Across Recordings

**H-reflex size per amplitude group** = MRA of the averaged bipolar waveform in the H-wave window, minus the pre-stimulus background MRA:

> **size (µV) = mean|avg_bip(t ∈ [H_START, H_END])| − mean|avg_bip(t < 0)|**

One value per amplitude group; the box shows spread across amplitude groups.

In [ ]:
# ── H-Reflex Size Across Recordings: Comparison Plot ─────────────────────────
# Toggle between H-Reflex size, M-Wave size, and Background MRA (pre-stim EMG level).
# Re-run the pre-compute cell above if you change analysis configuration constants.
from ipywidgets import Dropdown, ToggleButtons, Output, VBox
from IPython.display import display as _disp

_xr_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'}
)
if _xr_stages:
    _xr_stage_d.value = next(iter(_xr_stages))

_xr_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background')],
    description='Metric:',
    style={'button_width': '160px'},
)

_xr_out = Output()

def _xr_refresh():
    sk = _xr_stage_d.value
    mt = _xr_metric_d.value
    if not sk:
        return
    with _xr_out:
        _xr_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache, RECORDING_DIRS, sk, _xr_stages, metric=mt)

_xr_stage_d.observe(lambda c: _xr_refresh(), names='value')
_xr_metric_d.observe(lambda c: _xr_refresh(), names='value')
_disp(VBox([_xr_stage_d, _xr_metric_d, _xr_out]))
_xr_refresh()

# Section 6c: Cross-Recording DataFrame & Calculations

`xr_df` is a flat per-trial DataFrame built from `_xr_cache`.  
Re-run this cell any time you re-run the pre-compute cell above.

In [ ]:
import pandas as pd

# ── Build per-trial DataFrame from raw EMG ────────────────────────────────────
# Note: _xr_cache now stores per-amplitude-group averaged-waveform MRA (matching
# the waveform viewer). This DataFrame is independent — it computes metrics
# per individual trial directly from raw EMG for trial-level analysis.
_rows = []
for _rec_lbl, _rec in _all_recordings.items():
    _sr_xr    = _rec['sample_rate'] or SAMPLE_RATE
    _ms_ps_xr = 1000.0 / _sr_xr
    _rec_s_xr = int(TRIAL_RECORD_MS * _sr_xr / 1000)
    for _sk, (_trials, _hdr, _emg_bl, _slbl) in _rec['stage_map'].items():
        for _i, _t in enumerate(_trials):
            try:
                _tm, _et, *_ = get_trial_window(
                    _t, PRE_AVG_MS, POST_AVG_MS,
                    ms_per_sample=_ms_ps_xr, record_samples=_rec_s_xr)
                _bg_m = _tm < 0
                _bg_v = float(np.nanmean(np.abs(_et[_bg_m]))) if _bg_m.any() else 0.0
                _mm   = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
                _hm   = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
                _mv   = float(np.nanmean(np.abs(_et[_mm]))) - _bg_v if _mm.any() else float('nan')
                _hv   = float(np.nanmean(np.abs(_et[_hm]))) - _bg_v if _hm.any() else float('nan')
            except Exception:
                _mv = _hv = _bg_v = float('nan')
            _rows.append({
                'recording': _rec_lbl,
                'stage':     _slbl,
                'trial':     _i + 1,
                'h_size_uv': _hv,
                'm_size_uv': _mv,
                'bg_mra_uv': _bg_v,
                'hm_ratio':  _hv / _mv if (_mv and _mv > 0 and not np.isnan(_mv)) else float('nan'),
            })

xr_df = pd.DataFrame(_rows)
print(f"xr_df: {len(xr_df)} rows  (per-trial rectified EMG metrics)")
print(f"  recordings: {xr_df['recording'].nunique()}  |  stages: {xr_df['stage'].nunique()}")
xr_df.head(10)

xr_df: 1243 rows  (per-trial rectified EMG metrics)
  recordings: 1  |  stages: 2


,recording,stage,trial,h_size_uv,m_size_uv,bg_mra_uv,hm_ratio
0,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),1,156.626419,517.256607,107.846970,0.302802
1,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),2,225.942234,1319.213993,38.053097,0.171270
2,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),3,521.495819,784.307220,172.060089,0.664913
3,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),4,61.421371,849.556671,145.385468,0.072298
4,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),5,431.260986,1244.125183,140.283508,0.346638
5,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),6,511.465736,1898.769936,46.108604,0.269367
6,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),7,383.023163,886.968292,209.960052,0.431834
7,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),8,-3.966248,919.377411,306.079132,-0.004314
8,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),9,853.321308,1837.443684,63.565594,0.464407
9,HRPILOT-17 UPC6,MH Recruitment Curve (.hrs1),10,1064.852783,1826.987183,121.926880,0.582846


In [ ]:
# ── Per-recording summary statistics ─────────────────────────────────────────
xr_summary = (
    xr_df.groupby('recording')
    .agg(
        n_trials      = ('trial',      'count'),
        h_size_mean   = ('h_size_uv',  'mean'),
        h_size_std    = ('h_size_uv',  'std'),
        m_size_mean   = ('m_size_uv',  'mean'),
        m_size_std    = ('m_size_uv',  'std'),
        bg_mra_mean   = ('bg_mra_uv',  'mean'),
        hm_ratio_mean = ('hm_ratio',   'mean'),
        hm_ratio_std  = ('hm_ratio',   'std'),
    )
    .round(3)
)
xr_summary

,n_trials,h_size_mean,h_size_std,m_size_mean,m_size_std,bg_mra_mean,hm_ratio_mean,hm_ratio_std
recording,,,,,,,,
HRPILOT-17 UPC6,1243,921.277,551.972,2272.103,1091.155,264.119,0.469,1.883


In [ ]:
# ── M-wave size: pool first 4 recordings and compute grand mean ± SD ──────────
_baseline_recs = ['HRPILOT-17 CM1', 'HRPILOT-17 CM2', 'HRPILOT-17 CM3', 'HRPILOT-17 CM4', 'HRPILOT-17 CM5', 'HRPILOT-17 CM6', 'HRPILOT-17 CM7']

_pool = xr_df[xr_df['recording'].isin(_baseline_recs)]['m_size_uv'].dropna()

_grand_mean = _pool.mean()
_grand_std  = _pool.std(ddof=1)
_n_trials   = len(_pool)

print(f"M-wave size  (pooled across {_baseline_recs}):")
print(f"  N trials : {_n_trials}")
print(f"  Mean     : {_grand_mean:.3f} µV")
print(f"  SD       : {_grand_std:.3f} µV")
print()

# Per-recording breakdown for reference
print("Per-recording:")
for _rl in _baseline_recs:
    _d = xr_df[xr_df['recording'] == _rl]['m_size_uv'].dropna()
    print(f"  {_rl}: n={len(_d)}  mean={_d.mean():.3f}  SD={_d.std(ddof=1):.3f} µV")

M-wave size  (pooled across ['HRPILOT-17 CM1', 'HRPILOT-17 CM2', 'HRPILOT-17 CM3', 'HRPILOT-17 CM4', 'HRPILOT-17 CM5', 'HRPILOT-17 CM6', 'HRPILOT-17 CM7']):
  N trials : 0
  Mean     : nan µV
  SD       : nan µV

Per-recording:
  HRPILOT-17 CM1: n=0  mean=nan  SD=nan µV
  HRPILOT-17 CM2: n=0  mean=nan  SD=nan µV
  HRPILOT-17 CM3: n=0  mean=nan  SD=nan µV
  HRPILOT-17 CM4: n=0  mean=nan  SD=nan µV
  HRPILOT-17 CM5: n=0  mean=nan  SD=nan µV
  HRPILOT-17 CM6: n=0  mean=nan  SD=nan µV
  HRPILOT-17 CM7: n=0  mean=nan  SD=nan µV


# Section 6d: Trial Subset Filter

Filter every recording's trials by a per-trial EMG metric before analysis.  
Set `FILTER_TARGET` to your desired M-wave (or other) size, and `FILTER_TOLERANCE_PCT` to the ± window.  
Running the filter cell creates `_filtered_recordings` — a drop-in replacement for `_all_recordings` used by the comparison plot below.  
`xr_df_filtered` is the filtered subset of the DataFrame for calculations.

In [ ]:
# ── Trial Filter Configuration ─────────────────────────────────────────────────
# Metric to filter on: 'M_WAVE' | 'H_WAVE' | 'HM_RATIO' | 'BACKGROUND'
FILTER_METRIC        = 'M_WAVE'
FILTER_TARGET        = 2550      # µV  (or dimensionless ratio for HM_RATIO)
FILTER_TOLERANCE_PCT = 25.0       # ±%  around target

FILTER_LO = FILTER_TARGET * (1 - FILTER_TOLERANCE_PCT / 100)
FILTER_HI = FILTER_TARGET * (1 + FILTER_TOLERANCE_PCT / 100)
# Uncomment to override with manual bounds instead:
# FILTER_LO = 100.0
# FILTER_HI = 200.0

print(f"Metric  : {FILTER_METRIC}")
print(f"Target  : {FILTER_TARGET}  ±{FILTER_TOLERANCE_PCT}%")
print(f"Bounds  : [{FILTER_LO:.2f},  {FILTER_HI:.2f}]")

Metric  : M_WAVE
Target  : 2550  ±25.0%
Bounds  : [1912.50,  3187.50]


In [ ]:
# ── Apply per-trial filter ─────────────────────────────────────────────────────
# Computes the chosen metric from raw EMG for every trial, keeps those in [FILTER_LO, FILTER_HI].
# Creates:
#   _filtered_recordings  — same structure as _all_recordings, with only passing trials
#   xr_df_filtered        — filtered subset of xr_df for DataFrame calculations

_filtered_recordings = {}
for _rec_lbl, _rec in _all_recordings.items():
    _sr_f    = _rec['sample_rate'] or SAMPLE_RATE
    _ms_ps_f = 1000.0 / _sr_f
    _rec_s_f = int(TRIAL_RECORD_MS * _sr_f / 1000)
    _filt_sm = {}
    for _sk, (_trials, _hdr, _emg_bl, _slbl) in _rec['stage_map'].items():
        _keep = []
        for _t in _trials:
            try:
                _tm, _et, *_ = get_trial_window(
                    _t, PRE_AVG_MS, POST_AVG_MS,
                    ms_per_sample=_ms_ps_f, record_samples=_rec_s_f,
                )
                _bg_m = _tm < 0
                _bg_v = float(np.nanmean(np.abs(_et[_bg_m]))) if _bg_m.any() else 0.0
                if FILTER_METRIC == 'M_WAVE':
                    _wm = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
                    _val = float(np.nanmean(np.abs(_et[_wm]))) - _bg_v if _wm.any() else float('nan')
                elif FILTER_METRIC == 'H_WAVE':
                    _wm = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
                    _val = float(np.nanmean(np.abs(_et[_wm]))) - _bg_v if _wm.any() else float('nan')
                elif FILTER_METRIC == 'BACKGROUND':
                    _val = _bg_v
                elif FILTER_METRIC == 'HM_RATIO':
                    _mm = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
                    _hm = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
                    _mv = float(np.nanmean(np.abs(_et[_mm]))) - _bg_v if _mm.any() else float('nan')
                    _hv = float(np.nanmean(np.abs(_et[_hm]))) - _bg_v if _hm.any() else float('nan')
                    _val = (_hv / _mv) if (_mv and not np.isnan(_mv) and _mv > 0) else float('nan')
                else:
                    _val = float('nan')
            except Exception:
                _val = float('nan')
            if not np.isnan(_val) and FILTER_LO <= _val <= FILTER_HI:
                _keep.append(_t)
        _filt_sm[_sk] = (_keep, _hdr, _emg_bl, _slbl)
    _filtered_recordings[_rec_lbl] = {**_rec, 'stage_map': _filt_sm}

# ── Filtered DataFrame (from xr_df by metric column) ──────────────────────────
_filt_col = {
    'M_WAVE': 'm_size_uv', 'H_WAVE': 'h_size_uv',
    'HM_RATIO': 'hm_ratio', 'BACKGROUND': 'bg_mra_uv',
}.get(FILTER_METRIC, 'm_size_uv')
xr_df_filtered = xr_df[
    (xr_df[_filt_col] >= FILTER_LO) & (xr_df[_filt_col] <= FILTER_HI)
].copy().reset_index(drop=True)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"Filter: {FILTER_METRIC}  in  [{FILTER_LO:.2f}, {FILTER_HI:.2f}]\n")
_tot_orig = _tot_kept = 0
for _rl, _fr in _filtered_recordings.items():
    for _sk, (_ft, *_) in _fr['stage_map'].items():
        _orig = len(_all_recordings[_rl]['stage_map'][_sk][0])
        _kept = len(_ft)
        _tot_orig += _orig
        _tot_kept += _kept
        _pct = 100 * _kept / _orig if _orig else 0
        print(f"  {_rl}  [{_all_recordings[_rl]['stage_map'][_sk][3]}]:"
              f"  {_kept}/{_orig} trials  ({_pct:.1f}%)")
if _tot_orig:
    print(f"\n  Total: {_tot_kept}/{_tot_orig} trials  ({100*_tot_kept/_tot_orig:.1f}%)")
print(f"\nxr_df_filtered: {len(xr_df_filtered)} / {len(xr_df)} rows")

Filter: M_WAVE  in  [1912.50, 3187.50]

  HRPILOT-17 UPC6  [MH Recruitment Curve (.hrs1)]:  516/1242 trials  (41.5%)
  HRPILOT-17 UPC6  [Up Condition Pellet (.hrs4)]:  1/1 trials  (100.0%)

  Total: 517/1243 trials  (41.6%)

xr_df_filtered: 517 / 1243 rows


In [ ]:
# ── Filtered Comparison Plot + Waveform Viewer ────────────────────────────────
# Re-run this cell any time you change the filter bounds above.
from ipywidgets import Dropdown, ToggleButtons, Output, VBox
from IPython.display import display as _disp

_xr_cache_filt = compute_h_comparison_data(
    _filtered_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)

# ── Comparison boxplot ─────────────────────────────────────────────────────────
_xr_filt_out = Output()

def _xr_filt_render():
    _sk = _xr_filt_stage_d.value
    _mt = _xr_filt_metric_d.value
    if not _sk:
        return
    with _xr_filt_out:
        _xr_filt_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache_filt, RECORDING_DIRS, _sk, _xr_stages, metric=_mt)

_xr_filt_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'},
)
if _xr_stages:
    _xr_filt_stage_d.value = next(iter(_xr_stages))

_xr_filt_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background')],
    description='Metric:', style={'button_width': '160px'},
)

_xr_filt_stage_d.observe(lambda c: _xr_filt_render(), names='value')
_xr_filt_metric_d.observe(lambda c: _xr_filt_render(), names='value')
_disp(VBox([_xr_filt_stage_d, _xr_filt_metric_d, _xr_filt_out]))
_xr_filt_render()

# ── Filtered waveform analysis viewer ─────────────────────────────────────────
_fw_out = Output()

def _fw_stage_opts(rec_lbl):
    return [(lbl, sk)
            for sk, (_t, _h, _e, lbl) in _filtered_recordings[rec_lbl]['stage_map'].items()]

def _fw_render():
    _rl  = _fw_rec_d.value
    _sk2 = _fw_stage_d.value
    if not _rl or not _sk2:
        return
    _fr  = _filtered_recordings[_rl]
    _ft, _fh, _fe, _slbl = _fr['stage_map'][_sk2]
    _sr2 = _fr['sample_rate']
    _tp  = _apply_merge(_ft)
    with _fw_out:
        _fw_out.clear_output(wait=True)
        print(f'\n── Filtered Waveforms: {_slbl}  ({len(_ft)} filtered trials)  [{_rl}]')
        if not _ft:
            print('No trials pass the current filter for this recording/stage.')
            return
        plot_hrs2_analysis(
            _tp, _fh,
            pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
            n_per_page=N_PER_PAGE,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=_sr2,
            emg_blocks=_fe,
        )

_fw_rec_d = Dropdown(
    options=list(_filtered_recordings.keys()),
    value=next(iter(_filtered_recordings)),
    description='Recording:', layout={'width': '640px'},
)
_fw_stage_d = Dropdown(
    options=_fw_stage_opts(next(iter(_filtered_recordings))),
    description='Stage:', layout={'width': '480px'},
)

def _fw_on_rec(change):
    _fw_stage_d.unobserve(_fw_on_stage, names='value')
    _opts = _fw_stage_opts(_fw_rec_d.value)
    _fw_stage_d.options = _opts
    _fw_stage_d.value   = _opts[0][1] if _opts else None
    _fw_stage_d.observe(_fw_on_stage, names='value')
    _fw_render()

def _fw_on_stage(change):
    _fw_render()

_fw_rec_d.observe(_fw_on_rec,     names='value')
_fw_stage_d.observe(_fw_on_stage, names='value')

_fw_ctrl = VBox([_fw_rec_d, _fw_stage_d]) if len(_filtered_recordings) > 1 else VBox([_fw_stage_d])
_disp(VBox([_fw_ctrl, _fw_out]))
_fw_render()

Merged group ['0.066', '0.068', '0.070', '0.072', '0.074', '0.076', '0.078', '0.080', '0.081', '0.082', '0.083', '0.084', '0.086', '0.087', '0.088', '0.090', '0.091', '0.092', '0.093', '0.094', '0.095', '0.096', '0.097', '0.098', '0.099', '0.100', '0.101', '0.102', '0.103', '0.104', '0.105', '0.106', '0.107', '0.108', '0.109', '0.110', '0.111', '0.112', '0.113', '0.114', '0.115', '0.116', '0.117', '0.118', '0.119', '0.120', '0.121', '0.122', '0.123', '0.124', '0.125', '0.126', '0.127', '0.128', '0.129', '0.130', '0.131', '0.132', '0.133', '0.134', '0.135', '0.136', '0.137', '0.138', '0.139', '0.140', '0.141', '0.142', '0.143', '0.144', '0.145', '0.146', '0.147', '0.148', '0.149', '0.150', '0.152', '0.155', '0.156', '0.157', '0.158', '0.159', '0.160', '0.161', '0.162', '0.163', '0.164', '0.165', '0.166', '0.167', '0.168', '0.169', '0.170', '0.174', '0.175', '0.176', '0.177', '0.179', '0.180', '0.181', '0.182', '0.184', '0.185', '0.186', '0.187', '0.188', '0.189', '0.191', '0.193', '0.19

In [ ]:
'''
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────
from ipywidgets import Output, VBox
from IPython.display import display as _disp

_trv_out = Output()

def _trv_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _trv_get_state()
    _tp = _apply_merge(_st)
    with _trv_out:
        _trv_out.clear_output(wait=True)
        print(f'\n── Trial Viewer: {_slbl}  ({len(_st)} trials)  [{_rec}]')
        plot_hrs2_trials(
            _tp, _sh,
            pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
            n_per_page=N_PER_PAGE,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=_sr,
            emg_blocks=_se,
        )

_trv_ctrl, _trv_get_state, _trv_sync = _make_viewer_selector(_trv_refresh)
_refresh_viewers['trial_viewer'] = _trv_sync
_disp(VBox([_trv_ctrl, _trv_out]))
_trv_refresh()
'''


"\n# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────\nfrom ipywidgets import Output, VBox\nfrom IPython.display import display as _disp\n\n_trv_out = Output()\n\ndef _trv_refresh():\n    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _trv_get_state()\n    _tp = _apply_merge(_st)\n    with _trv_out:\n        _trv_out.clear_output(wait=True)\n        print(f'\n── Trial Viewer: {_slbl}  ({len(_st)} trials)  [{_rec}]')\n        plot_hrs2_trials(\n            _tp, _sh,\n            pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,\n            n_per_page=N_PER_PAGE,\n            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,\n            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,\n            sample_rate=_sr,\n            emg_blocks=_se,\n        )\n\n_trv_ctrl, _trv_get_state, _trv_sync = _make_viewer_selector(_trv_refresh)\n_refresh_viewers['trial_viewer'] = _trv_sync\n_disp(VBox([_trv_ctrl, _trv_out]))\n_trv_refresh()\n"

# Section 3c: H:M Ratio Summary

Box plot and histogram of H:M ratio for each stimulation polarity group.
If both normal and reversed polarities were used in this session, each group is analysed separately.

In [ ]:
# ── Stim Polarity Analysis ────────────────────────────────────────────────────
from ipywidgets import Output, VBox
from IPython.display import display as _disp

_pol_out = Output()

def _pol_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _pol_get_state()
    with _pol_out:
        _pol_out.clear_output(wait=True)
        print(f'\n── Polarity / H:M Ratio: {_slbl}  ({len(_st)} trials)  [{_rec}]')
        trials_by_polarity = split_trials_by_polarity(_st)
        plot_hm_ratio_summary(
            trials_by_polarity, _sh,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=_sr,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        )
        plot_hwave_regression(
            _st, _se,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=_sr,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        )

_pol_ctrl, _pol_get_state, _pol_sync = _make_viewer_selector(_pol_refresh)
_refresh_viewers['stim_polarity'] = _pol_sync
_disp(VBox([_pol_ctrl, _pol_out]))
_pol_refresh()


# Section 3d: M-Wave Stabilization Control Error (V3)

Trial-by-trial plot of the M-wave stabilization controller output.
- **Left axis** — `m_wave_error` (µV); falls back to `m_wave_window_median` when controller inactive.
- **Right axis** — `stimulation_amplitude_ma` (mA, orange).

Requires V3 recordings (S2 file_version ≥ 9; S4/S5/S6 file_version ≥ 2).

In [ ]:
# ── M-Wave Control Error with Reference Lines ─────────────────────────────────
# Controls:
#   Auto checkbox  — reads target M-size from the recording's stored set-value
#   Target (µV)    — manual override (editable when Auto is unchecked)
#   ±Inner %       — tolerance band drawn as green dashed lines around the target
#   ±Outer %       — algorithm bounds drawn as red dotted lines (set to 0 to hide)
#   Update Plot    — re-render with current settings after changing parameters
from ipywidgets import Output, VBox, HBox, Checkbox, FloatText, FloatSlider, Button, Label
from IPython.display import display as _disp
import numpy as np

_mw_out = Output()

# ── Control widgets ────────────────────────────────────────────────────────────
_mw_auto_cb   = Checkbox(value=True, description='Auto (recording set-value)',
                          style={'description_width': 'initial'}, layout={'width': '270px'})
_mw_target_ft = FloatText(value=0.0, description='Target (µV):',
                           step=10, disabled=True, layout={'width': '190px'})
_mw_inner_sl  = FloatSlider(value=25.0, min=0, max=100, step=1,
                              description='±Inner %:', readout_format='.0f',
                              style={'description_width': '70px'}, layout={'width': '280px'})
_mw_outer_sl  = FloatSlider(value=50.0, min=0, max=200, step=1,
                              description='±Outer %:', readout_format='.0f',
                              style={'description_width': '70px'}, layout={'width': '280px'})
_mw_apply_btn = Button(description='Update Plot', button_style='primary',
                        layout={'width': '120px'})
_mw_info_lbl  = Label(value='')

_mw_controls = VBox([
    HBox([_mw_auto_cb, _mw_target_ft, _mw_info_lbl]),
    HBox([_mw_inner_sl, _mw_outer_sl, _mw_apply_btn]),
])

def _mw_auto_target(trials):
    vals  = [getattr(t, 'm_wave_set_value_uv', float('nan')) for t in trials]
    valid = [v for v in vals if not np.isnan(v)]
    return float(np.median(valid)) if valid else float('nan')

def _mw_refresh():
    _sk, _st, _sh, _se, _slbl, _sr, _rec, _h1 = _mw_get_state()
    _local_sm  = _all_recordings[_rec]['stage_map']
    _mw_stages = {k: v for k, v in _local_sm.items()
                  if v[0] and any(
                      not np.isnan(getattr(t, 'm_wave_error', float('nan'))) or
                      not np.isnan(getattr(t, 'm_wave_window_median', float('nan')))
                      for t in v[0])}
    with _mw_out:
        _mw_out.clear_output(wait=True)
        if not _mw_stages:
            print("No M-wave stabilization data available in any loaded stage "
                  "(requires V3 S2 file_version ≥ 9, or S4/S5/S6 file_version ≥ 2).")
            return
        _key = _sk if _sk in _mw_stages else next(iter(_mw_stages))
        _mw_st, _mw_sh, _mw_se, _mw_slbl = _mw_stages[_key]

        if _mw_auto_cb.value:
            _auto = _mw_auto_target(_mw_st)
            if not np.isnan(_auto):
                _mw_target_ft.value = _auto
                _mw_info_lbl.value  = f'Auto: {_auto:.1f} µV'
            else:
                _mw_info_lbl.value  = 'Auto: no set-value stored'
            _target = None
        else:
            _target = _mw_target_ft.value
            _mw_info_lbl.value = f'Override: {_target:.1f} µV'

        print(f'\n── M-Wave Control Error: {_mw_slbl}  ({len(_mw_st)} trials)  [{_rec}]')
        plot_mwave_control_error(
            _mw_st, _mw_sh,
            sample_rate=_sr,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
            target_uv=_target,
            inner_pct=_mw_inner_sl.value,
            outer_pct=_mw_outer_sl.value,
        )

def _mw_auto_toggle(change):
    _mw_target_ft.disabled = change['new']
    _mw_refresh()

_mw_auto_cb.observe(_mw_auto_toggle, names='value')
_mw_apply_btn.on_click(lambda b: _mw_refresh())

_mw_ctrl, _mw_get_state, _mw_sync = _make_viewer_selector(_mw_refresh)
_refresh_viewers['mwave_control_error'] = _mw_sync

_disp(_mw_controls)
_disp(VBox([_mw_ctrl, _mw_out]))
_mw_refresh()

# Section 3e: Frequency Test Analysis (V3 FT)

Interactive viewer for `.hrsft` pulse-train data.  Layout mirrors the HRS2 Analysis viewer above.

**Navigation row** — `Prev` / `Next` / `Trial` dropdown navigate one trial at a time.  `Amp` dropdown filters to only trials at a specific stimulation amplitude (or "All").  `Freq` label shows the train frequency from the file header.

**Style row** — Controls how pulse traces are coloured and labelled:
- **Gradient** (default) — coolwarm colormap, blue = pulse 1 → red = last pulse.
- **Bold Ends** — same gradient, but first and last pulse are drawn thick so they stand out from the middle pulses.
- **Distinct** — tab20 qualitative palette, every pulse a unique colour (forces Labeled legend).
- **Show Legend** checkbox — toggle legend on/off.
- **Colorbar / Labeled** — gradient colorbar on the right axis, or individual per-pulse line entries in the plot legend.

**Pulse / zoom row** — `Pulse` dropdown highlights one specific pulse bold (lw 3.5, full alpha) and fades all others (lw 0.8, 15% alpha), making it easy to compare a single pulse against the rest of the train.  `Back to All` resets to equal weight.

**Controls row** — `Auto Y` checkbox auto-scales the y-axis; uncheck to type exact `Y min` / `Y max` bounds.  `Fig W` / `Fig H` resize the waveform figure (use when labels are getting clipped).  `Pre ms` / `Post ms` set the x-axis window around each pulse onset.

**Below the waveform viewer** — two static figures that update with every trial navigation:
- **H/M MRA Per Pulse** — mean rectified average (pre-stored app values) of the H-wave and M-wave per pulse across the train.  Shows homosynaptic (rate-dependent) depression.
- **H/M Peak Per Pulse** — max |EMG| within each wave window per pulse, computed from raw `trial_data`.

M/H wave windows are shared with the HRS2 configuration constants (`M_WAVE_START_MS`, `H_WAVE_START_MS`, etc.).

In [ ]:
# ── Frequency Test Analysis: Interactive Viewer ─────────────────────────────
from IPython.display import display as _disp
_disp(make_ft_viewer(
    _all_recordings, POST_PLOT_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
))

Label(value='No Frequency Test data loaded (.hrft not found in any recording directory).')